In [12]:
# How big is the "danger zone" segment?
danger_zone = df[
    (df['Contract'] == 'Month-to-month') & 
    (df['InternetService'] == 'Fiber optic')
]

print(f"Danger zone segment size: {len(danger_zone):,} customers ({len(danger_zone)/len(df):.1%} of base)")
print(f"Danger zone churn rate: {(danger_zone['Churn']=='Yes').mean():.1%}")
print(f"Danger zone monthly revenue: ${danger_zone['MonthlyCharges'].sum():,.0f}")
print(f"Danger zone monthly revenue AT RISK: ${danger_zone[danger_zone['Churn']=='Yes']['MonthlyCharges'].sum():,.0f}")
print(f"Annualized revenue at risk from this segment alone: ${danger_zone[danger_zone['Churn']=='Yes']['MonthlyCharges'].sum() * 12:,.0f}")

# What % of total churn revenue comes from this one segment?
total_churn_revenue = df[df['Churn']=='Yes']['MonthlyCharges'].sum()
segment_churn_revenue = danger_zone[danger_zone['Churn']=='Yes']['MonthlyCharges'].sum()
print(f"\nThis ONE segment accounts for {segment_churn_revenue/total_churn_revenue:.1%} of all churn revenue")

Danger zone segment size: 2,128 customers (30.2% of base)
Danger zone churn rate: 54.6%
Danger zone monthly revenue: $185,181
Danger zone monthly revenue AT RISK: $100,482
Annualized revenue at risk from this segment alone: $1,205,784

This ONE segment accounts for 72.2% of all churn revenue


In [11]:
print("Churn rate by Contract × Internet Service:")
print(churn_breakdown.map(lambda x: f"{x:.1%}"))

print("\nAvg monthly charge by Contract × Internet Service:")
print(charge_breakdown.map(lambda x: f"${x:.2f}"))

Churn rate by Contract × Internet Service:
InternetService    DSL Fiber optic     No
Contract                                 
Month-to-month   32.2%       54.6%  18.9%
One year          9.3%       19.3%   2.5%
Two year          1.9%        7.2%   0.8%

Avg monthly charge by Contract × Internet Service:
InternetService     DSL Fiber optic      No
Contract                                   
Month-to-month   $50.22      $87.02  $20.41
One year         $61.40      $98.78  $20.82
Two year         $70.46     $104.57  $21.78


In [1]:
import pandas as pd
print("pandas imported, version:", pd.__version__)

df = pd.read_csv('../data/raw/telco_churn_raw.csv')
print("df loaded, shape:", df.shape)

pandas imported, version: 3.0.3
df loaded, shape: (7043, 21)


In [2]:
print(df.groupby('Contract')['Churn'].count())

Contract
Month-to-month    3875
One year          1473
Two year          1695
Name: Churn, dtype: int64


In [3]:
# Cross-tab: churn rate by contract type AND internet service
churn_breakdown = df.groupby(['Contract', 'InternetService'])['Churn'].apply(
    lambda x: (x == 'Yes').mean()
).unstack()

print("Churn rate by Contract × Internet Service:")
print(churn_breakdown.map(lambda x: f"{x:.1%}"))

# Average monthly charge by these segments
charge_breakdown = df.groupby(['Contract', 'InternetService'])['MonthlyCharges'].mean().unstack()
print("\nAvg monthly charge by Contract × Internet Service:")
print(charge_breakdown.map(lambda x: f"${x:.2f}"))

Churn rate by Contract × Internet Service:
InternetService    DSL Fiber optic     No
Contract                                 
Month-to-month   32.2%       54.6%  18.9%
One year          9.3%       19.3%   2.5%
Two year          1.9%        7.2%   0.8%

Avg monthly charge by Contract × Internet Service:
InternetService     DSL Fiber optic      No
Contract                                   
Month-to-month   $50.22      $87.02  $20.41
One year         $61.40      $98.78  $20.82
Two year         $70.46     $104.57  $21.78


In [4]:
# Revenue at risk
total_monthly_revenue = df['MonthlyCharges'].sum()
churned_monthly_revenue = df[df['Churn'] == 'Yes']['MonthlyCharges'].sum()
retained_monthly_revenue = df[df['Churn'] == 'No']['MonthlyCharges'].sum()

print(f"Total monthly revenue:     ${total_monthly_revenue:>12,.0f}")
print(f"Revenue from retained:     ${retained_monthly_revenue:>12,.0f}")
print(f"Revenue lost to churn:     ${churned_monthly_revenue:>12,.0f}")
print(f"\nChurn = {churned_monthly_revenue/total_monthly_revenue:.1%} of monthly revenue")
print(f"Annualized churn impact:   ${churned_monthly_revenue * 12:,.0f}")

# Average customer value
print(f"\nAvg monthly charge (churned):  ${df[df['Churn']=='Yes']['MonthlyCharges'].mean():.2f}")
print(f"Avg monthly charge (retained): ${df[df['Churn']=='No']['MonthlyCharges'].mean():.2f}")

Total monthly revenue:     $     456,117
Revenue from retained:     $     316,986
Revenue lost to churn:     $     139,131

Churn = 30.5% of monthly revenue
Annualized churn impact:   $1,669,570

Avg monthly charge (churned):  $74.44
Avg monthly charge (retained): $61.27


In [5]:
# Convert TotalCharges to numeric, blanks become NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# For brand-new customers (tenure=0), TotalCharges should be 0
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Verify the fix
print("TotalCharges dtype:", df['TotalCharges'].dtype)
print("Missing values:", df['TotalCharges'].isnull().sum())
print(f"Range: ${df['TotalCharges'].min():.2f} to ${df['TotalCharges'].max():,.2f}")

TotalCharges dtype: float64
Missing values: 0
Range: $0.00 to $8,684.80


In [6]:
# Investigate TotalCharges
print("TotalCharges dtype:", df['TotalCharges'].dtype)
print("\nUnique non-numeric values (sample):")

# Try to convert and find which values fail
problem_mask = pd.to_numeric(df['TotalCharges'], errors='coerce').isnull()
print(f"\nNumber of problematic rows: {problem_mask.sum()}")
print(f"\nLet's look at these rows:")
df[problem_mask][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']]

TotalCharges dtype: float64

Unique non-numeric values (sample):

Number of problematic rows: 0

Let's look at these rows:


,customerID,tenure,MonthlyCharges,TotalCharges,Churn


In [7]:
# Bucket customers by tenure
df['tenure_bucket'] = pd.cut(
    df['tenure'], 
    bins=[0, 6, 12, 24, 48, 100],
    labels=['0-6 months', '7-12 months', '13-24 months', '25-48 months', '49+ months']
)

tenure_churn = df.groupby('tenure_bucket', observed=True)['Churn'].apply(
    lambda x: (x == 'Yes').mean()
).sort_values(ascending=False)

print("Churn rate by tenure:")
for bucket, rate in tenure_churn.items():
    print(f"  {bucket:20} {rate:.1%}")

Churn rate by tenure:
  0-6 months           53.3%
  7-12 months          35.9%
  13-24 months         28.7%
  25-48 months         20.4%
  49+ months           9.5%


In [8]:
# Churn rate by contract type
contract_churn = df.groupby('Contract')['Churn'].apply(
    lambda x: (x == 'Yes').mean()
).sort_values(ascending=False)

print("Churn rate by contract type:")
for contract, rate in contract_churn.items():
    print(f"  {contract:25} {rate:.1%}")

Churn rate by contract type:
  Month-to-month            42.7%
  One year                  11.3%
  Two year                  2.8%


In [9]:
# Data types and a peek at the data
print(df.dtypes)
print("\n" + "="*50 + "\n")
df.head()

customerID               str
gender                   str
SeniorCitizen          int64
Partner                  str
Dependents               str
tenure                 int64
PhoneService             str
MultipleLines            str
InternetService          str
OnlineSecurity           str
OnlineBackup             str
DeviceProtection         str
TechSupport              str
StreamingTV              str
StreamingMovies          str
Contract                 str
PaperlessBilling         str
PaymentMethod            str
MonthlyCharges       float64
TotalCharges         float64
Churn                    str
tenure_bucket       category
dtype: object




,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_bucket
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0-6 months
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,No,One year,No,Mailed check,56.95,1889.50,No,25-48 months
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,0-6 months
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,25-48 months
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,0-6 months


In [10]:
import pandas as pd
import numpy as np

# Load the data
df = pd.read_csv('../data/raw/telco_churn_raw.csv')
# Basic info
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"\nMissing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Churn rate
churn_rate = (df['Churn'] == 'Yes').mean()
print(f"\nOverall churn rate: {churn_rate:.1%}")

Rows: 7,043
Columns: 21

Missing values per column:
Series([], dtype: int64)

Overall churn rate: 26.5%
